In [ ]:
# install libraries #
!pip install pandas numpy matplotlib seaborn scikit-learn

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [3]:
# Load the dataset
data = pd.read_csv(r'C:\Users\woro_\OneDrive\DS_Projects\ML_Fraud_Detection_System\data\nova_pay_combined.csv')

In [4]:
# Explore the variable columns in the dataset.
pd.set_option('display.max_columns', None)
data.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,device_id,new_device,ip_address,ip_country,location_mismatch,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,1.351351,9f292dcc-3297-4947-a260-6a1ef69041ff,False,221.78.171.180,US,False,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,12.758621,3a95b9f5-309f-4684-a46d-e2ff2435bf78,True,120.12.20.29,CA,False,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,7.142857,a4737752-9aac-43ed-9d8b-2ccdffc24052,False,223.96.181.93,US,False,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,0.925926,6aeb85a3-5603-4221-896c-9e6882764f1a,False,186.228.15.74,US,False,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,83.333333,a5b9250e-dbe0-4c5f-a6e7-5492b7349402,False,11.82.47.62,US,False,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [5]:
# Explore the distribution of the target variable. Examine the skewness of the data and determine if there is a class imbalance issue.
# In this case there is a class imbalance.
data['is_fraud'].value_counts()

is_fraud
0    10403
1      997
Name: count, dtype: int64

In [6]:
# Check for missing values in the dataset and examine the data types of each variable to identify any potential issues
# data type mismatch.
pd.reset_option('display.max_rows')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  str    
 1   customer_id                11400 non-null  str    
 2   timestamp                  11371 non-null  str    
 3   home_country               11400 non-null  str    
 4   source_currency            11400 non-null  str    
 5   dest_currency              11400 non-null  str    
 6   channel                    11400 non-null  str    
 7   amount_src                 11400 non-null  str    
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  str    
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  str    
 14  i

In [7]:
# Convert timestamp to datetime format
data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')

print(data['timestamp'].head(5))
print(data['timestamp'].dtype)
print("missing timestamps:", data['timestamp'].isna().sum())

0   2022-10-03 18:40:59.468549+00:00
1   2022-10-03 20:39:38.468549+00:00
2   2022-10-03 23:02:43.468549+00:00
3   2022-10-04 01:08:53.468549+00:00
4   2022-10-04 09:35:03.468549+00:00
Name: timestamp, dtype: datetime64[us, UTC]
datetime64[us, UTC]
missing timestamps: 61


In [8]:
# Commas in amount_src are strings and need to be removed before converting to numerical format
data['amount_src'] = data['amount_src'].str.replace(',', '', regex=True)
data['amount_src'] = pd.to_numeric(data['amount_src'])
data['amount_src'].head()

0    278.19
1    208.51
2    160.33
3     59.41
4    200.96
Name: amount_src, dtype: float64

In [9]:
# drop identifier columns (They have no predictive power and they cause overfitting)
# List all the ID columns you want to remove
identifiers = ['transaction_id', 'customer_id', 'device_id']

# Drop all identifiers
data.drop(columns=identifiers, inplace=True)

# Confirm
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   timestamp                  11339 non-null  datetime64[us, UTC]
 1   home_country               11400 non-null  str                
 2   source_currency            11400 non-null  str                
 3   dest_currency              11400 non-null  str                
 4   channel                    11400 non-null  str                
 5   amount_src                 11400 non-null  float64            
 6   amount_usd                 11095 non-null  float64            
 7   fee                        11105 non-null  float64            
 8   exchange_rate_src_to_dest  11400 non-null  float64            
 9   new_device                 11400 non-null  bool               
 10  ip_address                 11095 non-null  str                
 11  ip_country   